# Freight Rate Forecasting
This notebook demonstrates the end-to-end pipeline for forecasting dry-bulk freight rates using chronological time-series validation.

**Disclaimer:** The data used in this notebook is synthetic and is intended only for demonstrating the modeling pipeline. Accuracy claims do not reflect real-world data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings('ignore')

# Ensure we are in the root directory
if os.getcwd().endswith('notebooks'):
    os.chdir('..')

from src.models.baseline_forecaster import BaselineForecaster
from src.models.arima_forecaster import ARIMAForecaster
from src.models.xgboost_forecaster import XGBoostForecaster
from src.models.model_evaluation import compare_models, evaluate_forecast

## 1. Data Preparation & Feature Engineering
Since this is a demonstration, we will generate a synthetic daily freight rate series that exhibits some trend, seasonality, and noise. We'll use this dataset to train our models.

In [ ]:
def generate_synthetic_freight_data(start_date='2020-01-01', days=1500):
    dates = pd.date_range(start=start_date, periods=days)
    time = np.arange(days)
    
    # Trend
    trend = 10000 + time * 2.5
    
    # Seasonality (yearly ~ 365 days)
    seasonality = 2000 * np.sin(2 * np.pi * time / 365)
    
    # Noise
    noise = np.random.normal(0, 500, days)
    
    rates = trend + seasonality + noise
    return pd.DataFrame({'date': dates, 'rate': rates})

df = generate_synthetic_freight_data()
plt.figure(figsize=(12, 5))
plt.plot(df['date'], df['rate'], alpha=0.7)
plt.title('Synthetic Freight Rates (Indonesia -> Dhamra, Panamax)')
plt.ylabel('Rate (USD)')
plt.xlabel('Date')
plt.grid(True)
plt.show()

## 2. Train / Validation / Test Split
To prevent future-data leakage and avoid random shuffling, we use a strict chronological split. We will train on the earliest data, tune on the validation set, and test on the most recent data.

In [ ]:
# Chronological split: 70% Train, 15% Validation, 15% Test
n = len(df)
train_end = int(n * 0.7)
val_end = int(n * 0.85)

df_train = df.iloc[:train_end]
df_val = df.iloc[train_end:val_end]
df_test = df.iloc[val_end:]

plt.figure(figsize=(12, 5))
plt.plot(df_train['date'], df_train['rate'], label='Train')
plt.plot(df_val['date'], df_val['rate'], label='Validation')
plt.plot(df_test['date'], df_test['rate'], label='Test')
plt.title('Chronological Data Split')
plt.legend()
plt.grid(True)
plt.show()

## 3. Model Training & Evaluation
We evaluate 3 models:
1. **Baseline Model** (Naive Moving Average)
2. **ARIMA** (Statistical)
3. **XGBoost** (Machine Learning with lag features)

In [ ]:
results = []

# 1. Baseline Model
baseline = BaselineForecaster(window_size=7)
baseline.fit(df_train, target_col='rate', date_col='date')
baseline_metrics = baseline.evaluate(df_val, target_col='rate', date_col='date')
results.append(baseline_metrics)

# 2. ARIMA Model
arima = ARIMAForecaster(order=(5, 1, 0))
arima.fit(df_train, target_col='rate', date_col='date')
arima_metrics = arima.evaluate(df_val, target_col='rate', date_col='date')
results.append(arima_metrics)

# 3. XGBoost Model
# Using lags for 1 day, 7 days, 14 days, and 30 days
xgb_model = XGBoostForecaster(lags=[1, 2, 3, 7, 14, 30], n_estimators=100, max_depth=4, learning_rate=0.05)
xgb_model.fit(df_train, target_col='rate', date_col='date')
xgb_metrics = xgb_model.evaluate(df_val, target_col='rate', date_col='date')
results.append(xgb_metrics)

## 4. Model Comparison
We compare the validation metrics (MAE, RMSE, MAPE) across the three models.

In [ ]:
comparison_df = compare_models(results)
display(comparison_df)

## 5. Forecasting into the Future
Let's assume the XGBoost model performs well. We'll use it to forecast the next 7 days for our specific route, providing confidence estimates.

In [ ]:
forecast_result = xgb_model.predict(
    horizon_days=7, 
    context_df=df_val, # We use the end of validation as our context to predict into test
    target_col='rate',
    date_col='date',
    route='Indonesia -> Dhamra',
    vessel_type='Panamax'
)

print(f"Forecast Model Used: {forecast_result.model_used}")
print(f"Route: {forecast_result.route}")
print(f"Vessel Type: {forecast_result.vessel_type}")
print(f"Horizon: {forecast_result.horizon_days} days\n")

for pt in forecast_result.series:
    print(f"Date: {pt.date} | Predicted Rate: ${pt.predicted_rate:.2f} | 95% CI: [{pt.lower_ci:.2f}, {pt.upper_ci:.2f}]")

## 6. Model Persistence
Finally, we save the chosen model for future inference.

In [ ]:
os.makedirs('trained_models', exist_ok=True)
xgb_model.save('trained_models/xgboost_freight_forecaster.pkl')
print("Model saved successfully to trained_models/xgboost_freight_forecaster.pkl")